# Forcasting with foundation models

In [1]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
path = str(Path.cwd().parent)
print(path)
sys.path.insert(1, path)

import numpy as np
import pandas as pd
import skforecast

print(skforecast.__version__)

/Users/javier.escobar/code/github/skforecast
0.22.0


## Libraries

In [2]:
# Libraries
# ==============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skforecast.datasets import fetch_dataset
from skforecast.foundation import ForecasterFoundation, FoundationModel
from skforecast.model_selection import (
    TimeSeriesFold,
    backtesting_foundation
)
from skforecast.plot import set_dark_theme

In [3]:
# Plots
# ==============================================================================
import plotly.graph_objects as go
import plotly.io as pio
import plotly.offline as poff
pio.templates.default = "seaborn"
poff.init_notebook_mode(connected=True)
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({'font.size': 8})

# Warnings configuration
# ==============================================================================
import warnings
warnings.filterwarnings('once')

color = '\033[1m\033[38;5;208m' 
print(f"{color}Version skforecast: {skforecast.__version__}")
# print(f"{color}Version scikit-learn: {sklearn.__version__}")
# print(f"{color}Version lightgbm: {lightgbm.__version__}")
print(f"{color}Version pandas: {pd.__version__}")
print(f"{color}Version numpy: {np.__version__}")


Version skforecast: 0.22.0
Version pandas: 2.3.3
Version numpy: 2.4.4


In [3]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available:  {torch.backends.mps.is_available()}")

CUDA available: False
MPS available:  True


## Input data

In [4]:
# Data download
# ==============================================================================
data = fetch_dataset(name='vic_electricity', raw=True)
data.info()

# Data preparation
# ==============================================================================
data = data.copy()
data['Time'] = pd.to_datetime(data['Time'], format='%Y-%m-%dT%H:%M:%SZ')
data = data.set_index('Time')
data = data.asfreq('30min')
data = data.sort_index()
data.head(2)

# Aggregating in 1H intervals
# ==============================================================================
# The Date column is eliminated so that it does not generate an error when aggregating.
data = data.drop(columns="Date")
data = (
    data
    .resample(rule="h", closed="left", label="right")
    .agg({
        "Demand": "mean",
        "Temperature": "mean",
        "Holiday": "mean",
    })
)
data

# Split data into train-val-test
# ==============================================================================
data = data.loc['2012-01-01 00:00:00':'2014-12-30 23:00:00', :].copy()
end_train = '2013-12-31 23:59:00'
end_validation = '2014-11-30 23:59:00'
data_train = data.loc[: end_train, :].copy()
data_val   = data.loc[end_train:end_validation, :].copy()
data_test  = data.loc[end_validation:, :].copy()

print(f"Train dates      : {data_train.index.min()} --- {data_train.index.max()}  (n={len(data_train)})")
print(f"Validation dates : {data_val.index.min()} --- {data_val.index.max()}  (n={len(data_val)})")
print(f"Test dates       : {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_test)})")


╭──────────────────────────── vic_electricity ─────────────────────────────╮
│ Description:                                                             │
│ Half-hourly electricity demand for Victoria, Australia                   │
│                                                                          │
│ Source:                                                                  │
│ O'Hara-Wild M, Hyndman R, Wang E, Godahewa R (2022).tsibbledata: Diverse │
│ Datasets for 'tsibble'. https://tsibbledata.tidyverts.org/,              │
│ https://github.com/tidyverts/tsibbledata/.                               │
│ https://tsibbledata.tidyverts.org/reference/vic_elec.html                │
│                                                                          │
│ URL:                                                                     │
│ https://raw.githubusercontent.com/skforecast/skforecast-                 │
│ datasets/main/data/vic_electricity.csv                                   │
│                                                                          │
│ Shape: 52608 rows x 5 columns                                            │
╰──────────────────────────────────────────────────────────────────────────╯

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52608 entries, 0 to 52607
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Time         52608 non-null  object 
 1   Demand       52608 non-null  float64
 2   Temperature  52608 non-null  float64
 3   Date         52608 non-null  object 
 4   Holiday      52608 non-null  bool   
dtypes: bool(1), float64(2), object(2)
memory usage: 1.7+ MB
Train dates      : 2012-01-01 00:00:00 --- 2013-12-31 23:00:00  (n=17544)
Validation dates : 2014-01-01 00:00:00 --- 2014-11-30 23:00:00  (n=8016)
Test dates       : 2014-12-01 00:00:00 --- 2014-12-30 23:00:00  (n=720)


## ForecasterFoundation

In [5]:
# Create and train ForecasterFoundation
# ==============================================================================
estimator = FoundationModel(model_id="autogluon/chronos-2-small")
forecaster = ForecasterFoundation(estimator=estimator)
forecaster.fit(series=data_train["Demand"], exog=data_train[["Temperature", "Holiday"]])

print(f"Device map: {forecaster.estimator.adapter.device_map}")
forecaster

Device map: auto


==================== 
ForecasterFoundation 
==================== 
Model: autogluon/chronos-2-small 
Context length: 8192 
Series names: ['Demand'] 
Exogenous included: True 
Exogenous names: Temperature, Holiday 
Context range: {'Demand': [Timestamp('2012-01-01 00:00:00'), Timestamp('2013-12-31 23:00:00')]} 
Training index type: DatetimeIndex 
Training index frequency: <Hour> 
Creation date: 2026-04-14 16:13:00 
Last fit date: 2026-04-14 16:13:00 
Skforecast version: 0.22.0 
Python version: 3.12.13 
Forecaster id: None

Two methods can be used to predict the next n steps: `predict()` or `predict_interval()`. The argument `levels` is used to indicate for which series estimate predictions. If `None` all series will be predicted.

In [6]:
# Predictions and prediction intervals
# ==============================================================================
steps = 24

# Predictions for item_1
predictions_item_1 = forecaster.predict(steps=steps)
display(predictions_item_1.head(3))
print("")

# Interval predictions for item_1 and item_2
predictions_intervals = forecaster.predict_interval(
    steps    = steps,
    interval = 0.9
)
display(predictions_intervals)

/opt/homebrew/Caskroom/miniconda/base/envs/skforecast_py12/lib/python3.12/site-packages/einops/einops.py:847: SyntaxWarning: invalid escape sequence '\s'
  \sum_{c, d, g} x[a, b, c] * y[c, b, d] * z[a, g, k]


,level,pred
2014-01-01 00:00:00,Demand,3626.196045
2014-01-01 01:00:00,Demand,3726.625000
2014-01-01 02:00:00,Demand,3806.432129


,level,pred,lower_bound,upper_bound
2014-01-01 00:00:00,Demand,3626.196045,3492.449951,3778.884766
2014-01-01 01:00:00,Demand,3726.625000,3553.048828,3926.610352
2014-01-01 02:00:00,Demand,3806.432129,3610.736816,4043.083984
2014-01-01 03:00:00,Demand,3846.585449,3608.271973,4124.371582
2014-01-01 04:00:00,Demand,3883.401855,3637.121094,4179.540527
2014-01-01 05:00:00,Demand,3998.019043,3692.394287,4299.623047
2014-01-01 06:00:00,Demand,4130.403320,3825.096924,4453.519043
2014-01-01 07:00:00,Demand,4209.418457,3863.855469,4578.099609
2014-01-01 08:00:00,Demand,4162.874023,3828.880371,4492.421387
2014-01-01 09:00:00,Demand,4014.003418,3758.508545,4311.038574


## Compare CPU vs GPU

In [18]:
import time
import gc
import torch

benchmark_results = []

In [19]:
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available:  {torch.backends.mps.is_available()}")

CUDA available: False
MPS available:  True


### Chronos-2 small

In [20]:
model_id = "autogluon/chronos-2-small"

# --- MPS (GPU) ---
estimator_mps = FoundationModel(model_id=model_id)
forecaster_mps = ForecasterFoundation(estimator=estimator_mps)
forecaster_mps.fit(series=data_train["Demand"], exog=data_train[["Temperature", "Holiday"]])

start = time.perf_counter()
preds_mps = forecaster_mps.predict(steps=24)
t_mps = time.perf_counter() - start
print(f"{model_id}")
print(f"MPS (GPU):  {t_mps:.4f}s")

del forecaster_mps, estimator_mps
gc.collect()
torch.mps.empty_cache()

# --- CPU ---
estimator_cpu = FoundationModel(model_id=model_id, device_map="cpu")
forecaster_cpu = ForecasterFoundation(estimator=estimator_cpu)
forecaster_cpu.fit(series=data_train["Demand"], exog=data_train[["Temperature", "Holiday"]])

start = time.perf_counter()
preds_cpu = forecaster_cpu.predict(steps=24)
t_cpu = time.perf_counter() - start
print(f"CPU:        {t_cpu:.4f}s")
print(f"Speedup:    {t_cpu / t_mps:.1f}x")

benchmark_results.append({"model_id": model_id, "t_mps": t_mps, "t_cpu": t_cpu})

del forecaster_cpu, estimator_cpu
gc.collect()

autogluon/chronos-2-small
MPS (GPU):  1.2386s
CPU:        1.1642s
Speedup:    0.9x


0

### TimesFM 2.5
TimesFM does not support exog. Device is auto-detected internally (CUDA > CPU, no MPS support).

In [21]:
model_id = "google/timesfm-2.5-200m-pytorch"

# TimesFM auto-detects device internally (CUDA > CPU, no MPS)
estimator_tfm = FoundationModel(model_id=model_id)
forecaster_tfm = ForecasterFoundation(estimator=estimator_tfm)
forecaster_tfm.fit(series=data_train["Demand"])

start = time.perf_counter()
preds_tfm = forecaster_tfm.predict(steps=24)
t_tfm = time.perf_counter() - start
print(f"{model_id}")
print(f"Time:       {t_tfm:.4f}s")

benchmark_results.append({"model_id": model_id, "t_mps": None, "t_cpu": t_tfm})

del forecaster_tfm, estimator_tfm
gc.collect()

google/timesfm-2.5-200m-pytorch
Time:       1.8751s


5649

### Moirai-2 small
Moirai does not support exog. Uses `device` parameter instead of `device_map`.

In [22]:
model_id = "Salesforce/moirai-2.0-R-small"

# --- MPS (GPU) ---
estimator_mps = FoundationModel(model_id=model_id, device="mps")
forecaster_mps = ForecasterFoundation(estimator=estimator_mps)
forecaster_mps.fit(series=data_train["Demand"])

try:
    start = time.perf_counter()
    preds_mps = forecaster_mps.predict(steps=24)
    t_mps = time.perf_counter() - start
    print(f"{model_id}")
    print(f"MPS (GPU):  {t_mps:.4f}s")
except RuntimeError as e:
    print(f"Error during MPS (GPU) prediction: {e}")
    t_mps = None

del forecaster_mps, estimator_mps
gc.collect()
torch.mps.empty_cache()

# --- CPU ---
estimator_cpu = FoundationModel(model_id=model_id, device="cpu")
forecaster_cpu = ForecasterFoundation(estimator=estimator_cpu)
forecaster_cpu.fit(series=data_train["Demand"])

start = time.perf_counter()
preds_cpu = forecaster_cpu.predict(steps=24)
t_cpu = time.perf_counter() - start
print(f"CPU:        {t_cpu:.4f}s")
if t_mps is not None:
    print(f"Speedup:    {t_cpu / t_mps:.1f}x")

benchmark_results.append({"model_id": model_id, "t_mps": t_mps, "t_cpu": t_cpu})

del forecaster_cpu, estimator_cpu
gc.collect()

Error during MPS (GPU) prediction: The operator 'aten::_cummax_helper' is not currently implemented for the MPS device. If you want this op to be added in priority during the prototype phase of this feature, please comment on https://github.com/pytorch/pytorch/issues/77764. As a temporary fix, you can set the environment variable `PYTORCH_ENABLE_MPS_FALLBACK=1` to use the CPU as a fallback for this op. WARNING: this will be slower than running natively on MPS.
CPU:        0.3913s


2395

### Summary

In [23]:
import pandas as pd

df_benchmark = pd.DataFrame(benchmark_results)
df_benchmark["speedup"] = df_benchmark["t_cpu"] / df_benchmark["t_mps"]
df_benchmark["model"] = df_benchmark["model_id"].str.split("/").str[-1]
df_benchmark = df_benchmark[["model", "t_mps", "t_cpu", "speedup"]]
df_benchmark.columns = ["Model", "MPS (s)", "CPU (s)", "Speedup"]
df_benchmark.style.format(
    {"MPS (s)": "{:.4f}", "CPU (s)": "{:.4f}", "Speedup": "{:.1f}x"},
    na_rep="N/A"
)

,Model,MPS (s),CPU (s),Speedup
0,chronos-2-small,1.2386,1.1642,0.9x
1,timesfm-2.5-200m-pytorch,N/A,1.8751,N/A
2,moirai-2.0-R-small,N/A,0.3913,N/A


## Backtesting

In [ ]:
# Backtesting multiple time series
# ==============================================================================
cv = TimeSeriesFold(
        steps              = 24,
        initial_train_size = len(data.loc[:end_validation]),
        refit              = False
)

metrics_levels, backtest_predictions = backtesting_foundation(
    forecaster            = forecaster,
    series                = data['Demand'],
    exog                  = data[["Temperature", "Holiday"]],
    cv                    = cv,
    metric                = 'mean_absolute_error',
    add_aggregated_metric = True
)

print("Backtest metrics")
display(metrics_levels)
print("")
print("Backtest predictions")
backtest_predictions.head(4)


  0%|          | 0/30 [00:00<?, ?it/s]

Backtest metrics


,mean_absolute_error
0,158.69341



Backtest predictions


,fold,pred
Time,,
2014-12-01 00:00:00,0,5488.939941
2014-12-01 01:00:00,0,5431.977051
2014-12-01 02:00:00,0,5347.607422
2014-12-01 03:00:00,0,5256.606934


In [ ]:
# Plot predictions vs real value
# ======================================================================================
fig = go.Figure()
trace1 = go.Scatter(x=data_test.index, y=data_test['Demand'], name="test", mode="lines")
trace2 = go.Scatter(x=backtest_predictions.index, y=backtest_predictions['pred'], name="prediction", mode="lines")
fig.add_trace(trace1)
fig.add_trace(trace2)
fig.update_layout(
    title="Real value vs predicted in test data",
    xaxis_title="Date time",
    yaxis_title="Demand",
    width=800,
    height=400,
    margin=dict(l=20, r=20, t=35, b=20),
    legend=dict(orientation="h", yanchor="top", y=1.01, xanchor="left", x=0)
)
fig.show()